# 00a — Optionally define `mask_pixel` on raw data

Load one or several image IDs into a Napari stack. Move the `image` slider between acquisitions and paint a separate mask on every slice. Masks remain in raw detector coordinates and are centered only later by `01_FTH.ipynb`. Existing masks from compatible image IDs can be used as templates.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

BASEFOLDER = Path.cwd().resolve()
ROOT = BASEFOLDER
os.environ.setdefault('NUMBA_DISABLE_JIT', '1')
os.environ.setdefault('XDG_CACHE_HOME', str(ROOT / 'processed' / 'napari_cache'))
os.environ.setdefault('XDG_CONFIG_HOME', str(ROOT / 'processed' / 'napari_config'))
print('Project root:', ROOT)
print('Notebook kernel Python:', sys.executable)
if importlib.util.find_spec('napari') is None:
    print('Napari is missing in this kernel; installing napari[all]...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'napari[all]'])
try:
    import napari
    from napari.utils.colormaps import DirectLabelColormap
except ImportError as exc:
    if 'pydantic' not in str(exc).lower():
        raise
    print('Repairing Pydantic compatibility, then retrying Napari...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'pydantic>=2.12,<3'])
    import napari
    from napari.utils.colormaps import DirectLabelColormap
import matplotlib.pyplot as plt
import numpy as np
from qtpy.QtWidgets import QApplication
sys.path.insert(0, str(ROOT))
from library.data_loading import SextantsNexusLoader
from library.mask_store import MaskStore
%gui qt

In [ ]:
# Use one ID or all image IDs needed for stitching.
IMAGE_IDS = [95, 96]
# One dark ID per image ID. Use None when an image has no dark acquisition.
DARK_IDS = [None, None]  # Example: [90, 90]
if len(DARK_IDS) != len(IMAGE_IDS):
    raise ValueError('DARK_IDS must contain one entry per IMAGE_IDS entry')
# Optional templates: target image ID -> existing mask image ID.
INITIAL_MASK_IDS = {}  # Example: {590: 589}
RAW_FOLDER = Path('/home/experiences/sextants/com-sextants/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/')
RAW_FOLDER = Path('../COMET_20260902_Cocoons_Laser_raw/')
loader = SextantsNexusLoader(RAW_FOLDER)
mask_store = MaskStore(ROOT / 'processed' / 'mask_pixels')
frames = [loader.load(image_id) for image_id in IMAGE_IDS]
shapes = {frame.image.shape for frame in frames}
if len(shapes) != 1:
    raise ValueError(f'All images in one stack must have the same shape, got {shapes}')
raw_images = []
dark_images = []
corrected_images = []
for image_id, dark_id, frame in zip(IMAGE_IDS, DARK_IDS, frames):
    image = np.asarray(frame.image, dtype=float).copy()
    if dark_id is None:
        dark = np.zeros_like(image, dtype=float)
    else:
        dark = np.asarray(loader.load(dark_id).image, dtype=float)
        if dark.shape != image.shape:
            raise ValueError(f'Dark {dark_id} shape {dark.shape} != image {image_id} shape {image.shape}')
    raw_images.append(image)
    dark_images.append(dark)
    corrected_images.append(image - dark)
    print(f'image {image_id}: dark={dark_id if dark_id is not None else "none"}')
raw_image_stack = np.stack(raw_images)
dark_stack = np.stack(dark_images)
image_stack = np.stack(corrected_images)
initial_masks = []
for image_id, frame in zip(IMAGE_IDS, frames):
    template_id = INITIAL_MASK_IDS.get(image_id, image_id)
    exists = mask_store.exists(template_id)
    mask = (mask_store.load(template_id, frame.image.shape)
            if exists else np.zeros(frame.image.shape, np.uint8))
    initial_masks.append(mask)
    print(f'slice {len(initial_masks)-1}: image {image_id}, exposure={frame.exposure:g}, template={template_id if exists else "empty"}')
mask_stack = np.stack(initial_masks).astype(np.uint8)

In [ ]:
app = QApplication.instance() or QApplication([])
try:
    viewer.close()
except NameError:
    pass
viewer = napari.Viewer(title='mask_pixel raw image stack')
viewer.add_image(
    raw_image_stack, name='raw images', colormap='viridis', visible=False,
    contrast_limits=np.nanpercentile(raw_image_stack, (0, 100)),
)
viewer.add_image(
    dark_stack, name='dark images', colormap='gray', visible=False,
)
viewer.add_image(
    image_stack, name='dark-corrected images', colormap='viridis',
    contrast_limits=np.nanpercentile(image_stack, (0, 100)),
)
mask_layer = viewer.add_labels(
    mask_stack, name='mask_pixel', opacity=0.6,
    colormap=DirectLabelColormap(color_dict={
        0: np.array([0.0, 0.0, 0.0, 0.0]),
        1: np.array([1.0, 0.0, 0.0, 1.0]),
        None: np.array([1.0, 0.0, 0.0, 1.0]),
    }),
)
mask_layer.selected_label = 1
mask_layer.brush_size = 5
viewer.layers.selection.active = mask_layer
viewer.dims.axis_labels = ('image', 'y', 'x')
viewer.dims.ndisplay = 2
viewer.window.show()
viewer.window._qt_window.raise_()
viewer.window._qt_window.activateWindow()
app.processEvents()
print('Use the image-axis slider; the slice-to-image-ID mapping is printed above.')
print("Paint label 1 with the brush, fill bucket, or polygon tool; label 0 erases.")
print('When finished, run the save cell below.')

In [ ]:
edited_masks = (np.asarray(mask_layer.data) > 0).astype(np.uint8)
if edited_masks.shape != image_stack.shape:
    raise ValueError(f'Edited masks have shape {edited_masks.shape}, expected {image_stack.shape}')
for image_id, image, mask_pixel in zip(IMAGE_IDS, image_stack, edited_masks):
    saved_path = mask_store.save(image_id, mask_pixel)
    preview_path = saved_path.with_name(saved_path.stem + '_preview.png')
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image, cmap='viridis')
    axes[0].imshow(mask_pixel, alpha=0.35, cmap='Reds', vmin=0, vmax=1)
    axes[0].set_title(f'dark-corrected image {image_id} + mask')
    axes[1].imshow(mask_pixel, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title('mask_pixel (raw coordinates)')
    for axis in axes:
        axis.set_axis_off()
    fig.tight_layout()
    fig.savefig(preview_path, dpi=180, bbox_inches='tight')
    plt.close(fig)
    print(f'Image {image_id}: saved {mask_pixel.sum()} masked pixels to {saved_path}')
    print('  preview:', preview_path)